# multilingual-e5-large — preprocessing ablation

Tests, rather than assumes, whether two extra preprocessing steps actually improve clustering on
this corpus, holding embedding model (`multilingual-e5-large`), UMAP settings, and clustering
algorithm (Agglomerative/Ward — the best-evidenced algorithm from the literature review) all fixed.
Only the **text preprocessing** differs between the two arms:

- **Baseline**: the cleaning already used everywhere else in this project — NFC normalization,
  title-dedup, whitespace normalization. No stopword/lemmatization/stemming before embedding,
  per the literature (contextual transformer embeddings are trained on natural text; aggressive
  preprocessing tends not to help and can hurt — see the research summary in this project's
  conversation history).
- **Enhanced**: baseline + two additions, each with a specific evidence basis:
  1. **Grapheme-aware short-token filtering** — drop punctuation and any "word" with ≤2 grapheme
     clusters. Already validated in `notebooks/clustering_BGE_M3_BERTopic.ipynb`; important for
     Sinhala specifically because its combining-mark orthography means character count ≠ visual
     grapheme count (an "Enhanced Tokenizer for Sinhala" paper found language-specific,
     grapheme-aware handling meaningfully improves NLP task performance for this language).
  2. **Corpus-specific frequent-term removal** — drop the top ~75 most-frequent tokens *in this
     corpus* (not a generic stopword list). One clustering study found this specific technique
     improved clustering accuracy (53.8% → 55.3%), distinct from — and not contradicting — the
     finding that generic stopword removal doesn't help.

Both arms are embedded, UMAP-reduced, and clustered independently, then compared side-by-side on
the same metrics used throughout this project.

In [1]:
!pip install -q umap-learn grapheme

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 3.5 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [2]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
import string
from collections import Counter
from pathlib import Path
import grapheme

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_preprocessing_ablation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: /kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data
Results dir: /kaggle/working/results/e5_preprocessing_ablation


In [3]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

(750, 6) (750, 8) (800, 6)


In [4]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

df shape: (2000, 6)


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [5]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

df shape after required-field dropna: (1999, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   article_id    1999 non-null   object
 1   publisher     1999 non-null   object
 2   url           1999 non-null   object
 3   published_at  1999 non-null   object
 4   title         1999 non-null   object
 5   body_text     1999 non-null   object
dtypes: object(6)
memory usage: 109.3+ KB


In [6]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [7]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

,title,text
0,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,ස්ථාන දෙකකදී ඝාතන දෙකක්,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [8]:
# Whitespace normalization — this is the BASELINE cleaning shared by both arms
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after baseline text cleaning:", df.shape)
df.head()

df shape after baseline text cleaning: (1999, 7)


,article_id,publisher,url,published_at,title,body_text,text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...,"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [9]:
# Baseline document field (title + baseline-cleaned body), with the "passage: " prefix
# multilingual-e5 models require for corpus-side text
def build_passage_text(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    combined = f"{title}. {body}" if title else body
    return "passage: " + combined


df["passage_text_baseline"] = df.apply(lambda row: build_passage_text(row["title"], row["text"]), axis=1)
print(f"Documents: {len(df)}")
df["passage_text_baseline"].iloc[0][:500]

Documents: 1999


'passage: දැන් තෝරු-මෝරු අහුවෙන කාලේ. දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහනුවර දිස්ත්\u200dරික් මන්ත්\u200dරී ජගත් මනුවර්ණ මහතා:-(ජා.ජ.බ) පාර්ලිමේන්තුවේදී පැවසීය.\nදූෂණයට විරුද්ධ වීම භයානක බවත් දූෂණයට විරුද්ධ නොවී සිටීම ඊට වඩා භයානක බවත් අනුර දිසානායක ජනාධිපති\xa0 එක්සත් ජාතීන්ගේ මහා මණ්ඩලයේ අමතමින් ප්\u200dරකාශ කළා.එය අප නැවත අවධාරණය කළ යුතුයි.අපි දේශපාලන පලි ගැනීම් කරනවා යැයි චෝදනා කරනවා.නමුත් ඇත්ත ඒක නෙමෙයි.මත්තල ගුවන් තොටුපොලේ මගින් පර්යන්තය එක\xa0 ආණ්ඩුවක් නෙළුම් පොහොට්ටුවක හැඩයකට හදන්න තීරණය කළාම ඊට පස'

In [10]:
# Enhanced arm, step 1: grapheme-aware short-token filtering
# (matches notebooks/clustering_BGE_M3_BERTopic.ipynb\'s clean_text function)
def grapheme_filter(text):
    # Replace punctuation with spaces
    text = "".join(
        " " if unicodedata.category(char).startswith("P") else char
        for char in text
    )
    # Keep words with more than 2 graphemes
    return " ".join(
        word for word in text.split()
        if len(list(grapheme.graphemes(word))) > 2
    )


df["text_grapheme_filtered"] = df["text"].apply(grapheme_filter)
print(df["text_grapheme_filtered"].iloc[0][:500])

හාල්මැස්සො අහුවෙන කාලය බවමහනුවර දිස්ත්‍රික් මන්ත්‍රී ජගත් මනුවර්ණ මහතා පාර්ලිමේන්තුවේදී පැවසීය දූෂණයට විරුද්ධ භයානක බවත් දූෂණයට විරුද්ධ සිටීම භයානක බවත් අනුර දිසානායක ජනාධිපති එක්සත් ජාතීන්ගේ මණ්ඩලයේ අමතමින් ප්‍රකාශ නැවත අවධාරණය යුතුයි දේශපාලන ගැනීම් කරනවා චෝදනා කරනවා නමුත් ඇත්ත නෙමෙයි මත්තල ගුවන් තොටුපොලේ මගින් පර්යන්තය ආණ්ඩුවක් නෙළුම් පොහොට්ටුවක හැඩයකට හදන්න තීරණය කළාම පස්සේ ආණ්ඩුව අරලිය පෙත්තක හැඩයකට වෙනස් කරන්න තීරණය කරලා ඒවට ගෙවන්නෙ කවුද ජනතාවගේ සල්ලි මිනිස්සු පාරට ඇදගෙන වැටුනේ දේවල් එකට උත


In [11]:
# Enhanced arm, step 2: corpus-specific frequent-term removal
# (top ~75 most-frequent tokens BY DOCUMENT FREQUENCY in this corpus specifically — not a
# generic stopword list)
N_FREQUENT_TERMS_TO_DROP = 75

doc_freq = Counter()
for text in df["text_grapheme_filtered"]:
    doc_freq.update(set(text.split()))

most_frequent_terms = {term for term, _ in doc_freq.most_common(N_FREQUENT_TERMS_TO_DROP)}
print(f"Dropping {len(most_frequent_terms)} corpus-specific frequent terms, e.g.:")
print(sorted(most_frequent_terms)[:20])


def drop_frequent_terms(text):
    return " ".join(word for word in text.split() if word not in most_frequent_terms)


df["text_enhanced"] = df["text_grapheme_filtered"].apply(drop_frequent_terms)
df["passage_text_enhanced"] = df.apply(lambda row: build_passage_text(row["title"], row["text_enhanced"]), axis=1)
print(df["passage_text_enhanced"].iloc[0][:500])

Dropping 75 corpus-specific frequent terms, e.g.:
['අතර', 'අදහස්', 'අදාළ', 'අනතුරුව', 'අනුව', 'අමාත්\u200dය', 'අවශ්\u200dය', 'අවසන්', 'ආරම්භ', 'ඇත්තේ', 'ඇතැයි', 'ඇතුළු', 'ඉදිරිපත්', 'එහිදී', 'ඔවුන්', 'කටයුතු', 'කරන', 'කරන්න', 'කරනු', 'කරමින්']
passage: දැන් තෝරු-මෝරු අහුවෙන කාලේ. හාල්මැස්සො අහුවෙන කාලය බවමහනුවර දිස්ත්‍රික් මන්ත්‍රී ජගත් මනුවර්ණ පාර්ලිමේන්තුවේදී දූෂණයට විරුද්ධ භයානක දූෂණයට විරුද්ධ සිටීම භයානක අනුර දිසානායක එක්සත් ජාතීන්ගේ මණ්ඩලයේ අමතමින් නැවත අවධාරණය යුතුයි දේශපාලන ගැනීම් කරනවා චෝදනා කරනවා නමුත් ඇත්ත නෙමෙයි මත්තල ගුවන් තොටුපොලේ පර්යන්තය ආණ්ඩුවක් නෙළුම් පොහොට්ටුවක හැඩයකට හදන්න තීරණය කළාම පස්සේ ආණ්ඩුව අරලිය පෙත්තක හැඩයකට වෙනස් තීරණය කරලා ඒවට ගෙවන්නෙ කවුද ජනතාවගේ සල්ලි මිනිස්සු පාරට ඇදගෙන වැටුනේ දේවල් එකට උත්තර බඳින්න කලි


In [12]:
# Load multilingual-e5-large (the finalized embedding model for this project)
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
model.eval()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_passages(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# Embed both arms
embeddings_baseline = embed_passages(df["passage_text_baseline"].tolist(), batch_size=16)
print("baseline embeddings:", embeddings_baseline.shape)

embeddings_enhanced = embed_passages(df["passage_text_enhanced"].tolist(), batch_size=16)
print("enhanced embeddings:", embeddings_enhanced.shape)

baseline embeddings: (1999, 1024)
enhanced embeddings: (1999, 1024)


In [14]:
# Shared evaluation pipeline: separation score -> UMAP -> Agglomerative (Ward, threshold swept) -> metrics
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import AgglomerativeClustering
from umap import UMAP

rng = np.random.default_rng(SEED)


def evaluate_arm(arm_name, embeddings):
    sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
    sims = cosine_similarity(embeddings[sample_idx])
    pairwise = sims[np.triu_indices_from(sims, k=1)]
    separation_score = 1 - pairwise.mean()

    reduced = UMAP(
        n_neighbors=3, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED,
    ).fit_transform(embeddings)

    best = None
    for threshold in np.linspace(0.5, 15.0, 30):
        labels = AgglomerativeClustering(
            n_clusters=None, distance_threshold=threshold, linkage="ward",
        ).fit_predict(reduced)
        n_found = len(set(labels))
        if 1 < n_found < len(labels):
            sil = silhouette_score(reduced, labels, metric="cosine")
            if best is None or sil > best[1]:
                best = (threshold, sil, labels)

    threshold, sil, labels = best
    dbi = davies_bouldin_score(reduced, labels)
    ch = calinski_harabasz_score(reduced, labels)
    n_clusters = len(set(labels))

    row = {
        "arm": arm_name,
        "separation_score": round(separation_score, 4),
        "n_clusters": n_clusters,
        "distance_threshold": round(float(threshold), 3),
        "silhouette": round(sil, 4),
        "davies_bouldin": round(dbi, 4),
        "calinski_harabasz": round(ch, 2),
    }
    print(f"=== {arm_name} ===")
    for k, v in row.items():
        if k != "arm":
            print(f"  {k}: {v}")
    return row, labels


baseline_row, baseline_labels = evaluate_arm("baseline (project-standard cleaning)", embeddings_baseline)
enhanced_row, enhanced_labels = evaluate_arm("enhanced (+ grapheme filter + frequent-term removal)", embeddings_enhanced)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


=== baseline (project-standard cleaning) ===
  separation_score: 0.20909999310970306
  n_clusters: 330
  distance_threshold: 0.5
  silhouette: 0.8054999709129333
  davies_bouldin: 0.4009
  calinski_harabasz: 12028.330078125


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


=== enhanced (+ grapheme filter + frequent-term removal) ===
  separation_score: 0.1979999989271164
  n_clusters: 326
  distance_threshold: 0.5
  silhouette: 0.7832000255584717
  davies_bouldin: 0.4397
  calinski_harabasz: 11582.1103515625


In [15]:
# Side-by-side comparison
comparison_df = pd.DataFrame([baseline_row, enhanced_row])
comparison_df

,arm,separation_score,n_clusters,distance_threshold,silhouette,davies_bouldin,calinski_harabasz
0,baseline (project-standard cleaning),0.2091,330,0.5,0.8055,0.4009,12028.330078
1,enhanced (+ grapheme filter + frequent-term re...,0.1980,326,0.5,0.7832,0.4397,11582.110352


In [16]:
# Verdict + save
better_arm = comparison_df.sort_values("silhouette", ascending=False).iloc[0]["arm"]
print(f"Better arm by silhouette score: {better_arm}")
print()
print("Read this as a directional signal, not a final verdict — it's one embedding model, one")
print("clustering algorithm, and one corpus. If the gap is small, prefer the simpler baseline")
print("pipeline (fewer moving parts to maintain and explain).")

comparison_path = RESULTS_DIR / "e5_preprocessing_ablation_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"\nSaved comparison to {comparison_path}")

df["cluster_id_baseline"] = baseline_labels
df["cluster_id_enhanced"] = enhanced_labels
assignments_path = RESULTS_DIR / "e5_preprocessing_ablation_assignments.csv"
df.drop(columns=[c for c in df.columns if c.startswith("passage_text")], errors="ignore").to_csv(
    assignments_path, index=False, encoding="utf-8-sig"
)
print(f"Saved assignments (both arms) to {assignments_path}")

Better arm by silhouette score: baseline (project-standard cleaning)

Read this as a directional signal, not a final verdict — it's one embedding model, one
clustering algorithm, and one corpus. If the gap is small, prefer the simpler baseline
pipeline (fewer moving parts to maintain and explain).

Saved comparison to /kaggle/working/results/e5_preprocessing_ablation/e5_preprocessing_ablation_comparison.csv
Saved assignments (both arms) to /kaggle/working/results/e5_preprocessing_ablation/e5_preprocessing_ablation_assignments.csv


In [17]:
# Inspect a RANDOM sample of clusters from BOTH arms, side by side, for a qualitative read on
# whether the enhanced preprocessing actually changed anything a human would notice
rng_inspect = np.random.default_rng(SEED)

for label_col, arm_name in [("cluster_id_baseline", "BASELINE"), ("cluster_id_enhanced", "ENHANCED")]:
    print(f"########## {arm_name} ({label_col}) ##########\n")
    cluster_ids = df.loc[df[label_col] != -1, label_col].unique()
    sample_size = min(5, len(cluster_ids))
    sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)
    for cluster_id in sampled_cluster_ids:
        group = df[df[label_col] == cluster_id]
        print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
        for title in group["title"].head(15):
            print(f"- {title}")
        print()

########## BASELINE (cluster_id_baseline) ##########

=== Cluster 257 (4 articles) ===
- නාවික හමුදා සෙබළෙකු කැති පහරින් මරුට
- පොල් වත්තේදී ඝාතනයවූ තරුණයා මරණ පරීක්ෂණයේදී කාන්තාවක් වෙයි
- බිරිඳගේ හිස රැගෙන පොලීසියට ගිය අනිල්
- ගෑණු හුටපටයකට සෙබළෙක් සෙබළකු මරයි

=== Cluster 110 (7 articles) ===
- දේවානන්දා අද අධිකරණයට
- හිටපු රාජ්‍ය අමාත්‍ය රත්වත්තේගේ මරණ සහතිකය අධිකරණය ඉල්ලයි
- ලොහාන් රත්වත්තේ ගේ පෞද්ගලික ලේකම්වරයෙකුගේ සිරුර හමුවෙයි
- ලොහාන්ගේ බිරිඳ රිමාන්ඩ්
- ලොහාන් රත්වත්තේ අත්අඩංගුවට
- හිටපු අමාත්‍ය ඩග්ලස් දේවානන්දා රිමාන්ඩ්
- රත්වත්තේගේ මෝටර් රථයේ අංක තහඩුව ගැන අධිකරණයේදී එළියට ආ කථාව

=== Cluster 292 (6 articles) ===
- බ්‍රිතාන්‍යය ට නව අගමැතිවරයෙක්
- රාජ්‍ය පරිපාලන වැය ශීර්ෂය සම්මතයි
- ජනපති සර්ව පාක්ෂික හමුවක් කැඳවයි
- අයවැයට කැබිනට් අනුමැතිය
- ආණ්ඩු පක්ෂ මන්‍ත්‍රි කණ්ඩායමේ විශේෂ සාකච්ඡාවක් ජනපති හා අගමැති ප්‍රධානත්වයෙන්
- හදිසි කැබිනට් සංශෝධනයක්

=== Cluster 1 (20 articles) ===
- ජන අරගල සන්ධානය : 'ගෝල්ෆේස් අරගලකරුවන්' රැසක් සමග දේශපාලන කරලියට එන නව සන්ධානය
- මහ මැතිවරණයක් පැව

In [18]:
# Print every cluster (capped at 20 titles each, so this stays readable even when there are
# hundreds of clusters)
for cluster_id, group in sorted(
    df[df["cluster_id"] != -1].groupby("cluster_id"), key=lambda item: item[0]
):
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(20):
        print(f"- {title}")
    if len(group) > 20:
        print(f"... ({len(group) - 20} more)")
    print()


KeyError: 'cluster_id'